# Nível 1 — Carregamento e inspeção inicial

Nesta etapa, observamos os dados brutos sem remover, preencher, converter ou sinalizar registros.

In [1]:
import json

import pandas as pd

In [2]:
caminho_dados = '../dados/dados_nivel_1.json'

with open(caminho_dados, encoding='utf-8') as arquivo:
    dados_brutos = json.load(arquivo)

In [3]:
taxa_cambio_usd_brl = dados_brutos['taxa_cambio_usd_brl']
taxa_cambio_usd_brl

5.4

In [4]:
df_operacoes = pd.DataFrame(dados_brutos['operacoes'])

In [5]:
df_operacoes.head()

        id cliente_id  ...         contraparte  observacao
0  OP-0001    CLI-A-1  ...  Alfa Comercio LTDA            
1  OP-0002    CLI-A-1  ...  Alfa Comercio LTDA            
2  OP-0003    CLI-A-1  ...    Beta Servicos ME            
3  OP-0004    CLI-A-1  ...  Gama Distribuidora            
4  OP-0005    CLI-A-2  ...   Delta Transportes            

[5 rows x 9 columns]

In [6]:
quantidade_linhas, quantidade_colunas = df_operacoes.shape
print(f'Linhas: {quantidade_linhas}')
print(f'Colunas: {quantidade_colunas}')

Linhas: 20
Colunas: 9


In [7]:
df_operacoes.columns.tolist()

['id', 'cliente_id', 'data', 'valor', 'moeda', 'canal', 'tipo', 'contraparte', 'observacao']

In [8]:
df_operacoes.dtypes

id               str
cliente_id       str
data             str
valor          int64
moeda            str
canal            str
tipo             str
contraparte      str
observacao       str
dtype: object

In [9]:
df_operacoes.isna().sum()

id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

In [10]:
ids_duplicados = df_operacoes['id'].duplicated(keep=False)
print(f'Linhas com ID duplicado: {ids_duplicados.sum()}')
df_operacoes.loc[ids_duplicados, ['id']]

Linhas com ID duplicado: 2


        id
6  OP-0007
9  OP-0007

## Limpeza dos dados

A seguir, a base bruta é preservada e criamos uma versão tratada somente para remover duplicatas exatas e converter o tipo da coluna de data.

In [11]:
comparacao_op_0007 = df_operacoes.loc[df_operacoes['id'] == 'OP-0007'].reset_index(drop=True)
sao_duplicatas_exatas = (
    len(comparacao_op_0007) == 2
    and comparacao_op_0007.duplicated(keep=False).all()
)
print(f'As duas ocorrências de OP-0007 são duplicatas exatas: {sao_duplicatas_exatas}')
comparacao_op_0007.T

As duas ocorrências de OP-0007 são duplicatas exatas: True


                                 0                      1
id                         OP-0007                OP-0007
cliente_id                 CLI-A-3                CLI-A-3
data                    2026-03-05             2026-03-05
valor                        17200                  17200
moeda                          BRL                    BRL
canal                          pix                    pix
tipo         transferencia_enviada  transferencia_enviada
contraparte    Epsilon Consultoria    Epsilon Consultoria
observacao                                               

In [12]:
linhas_antes_remocao = len(df_operacoes)
df_operacoes_tratado = df_operacoes.drop_duplicates()
linhas_depois_remocao = len(df_operacoes_tratado)

print(f'Linhas antes da remoção: {linhas_antes_remocao}')
print(f'Linhas depois da remoção: {linhas_depois_remocao}')

Linhas antes da remoção: 20
Linhas depois da remoção: 19


In [13]:
df_operacoes_tratado['data'] = pd.to_datetime(df_operacoes_tratado['data'])
df_operacoes_tratado['data'].dtype

dtype('<M8[us]')

In [14]:
df_operacoes_tratado.isna().sum()

id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

### Decisão de limpeza

A duplicata exata de `OP-0007` foi removida para evitar que uma mesma operação seja contada duas vezes nas análises posteriores. Mantivemos uma ocorrência para preservar a operação legítima registrada na base.

O registro sem `data` foi mantido porque os demais campos ainda podem contribuir para análises que não dependem de data, como volume total e contagens gerais. Esse registro não poderá participar de regras que exigem agrupamento temporal, pois não inventamos nem preenchemos uma data ausente.